<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook05_Recover_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook closely follows algorithms written by Hsu et al. (2024)

In [14]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
print("Install complete.")
print("If Cell 3 throws a numpy ABI error: Runtime > Restart session + run all")

Install complete.
If Cell 3 throws a numpy ABI error: Runtime > Restart session + run all


In [15]:
# === Cell 2: Boilerplate ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime (A100 or L4)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


Importing libraries

In [16]:
import json
import random
import torch
import numpy as np
from pathlib import Path
import pandas as pd
from transformers import CLIPTextModel, CLIPTokenizer
import argparse

parameters of gene-algo

In [17]:
population_size = 200
generation = 3000
mutateRate = 0.25
crossoverRate = 0.5
length = 16
cof = 3
target_prompts = json.loads(Path("prompts/target_prompts.json").read_text())

Get concept vector (verbatim from repo)

In [20]:
dir_ = "CompVis/stable-diffusion-v1-4"
tokenizer = CLIPTokenizer.from_pretrained(dir_, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(dir_, subfolder="text_encoder").to('cuda')
num_samples = 5
df = pd.read_csv('prompts/vangogh_pairs.csv')
vangogh_text = []
for _, row in df.iterrows():
    prompt = [f"{row.van_gogh}"]*num_samples
    text_input = tokenizer(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
    embed = text_encoder(text_input.input_ids.to('cuda'), return_dict=True)[0]
    vangogh_text.extend(embed.detach().cpu().numpy())
vangogh_text = np.array(vangogh_text)

NoVangogh_text = []
df = pd.read_csv('prompts/vangogh_pairs.csv')
for _, row in df.iterrows():
    prompt = [f"{row.not_gogh}"]*num_samples
    text_input = tokenizer(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
    embed = text_encoder(text_input.input_ids.to('cuda'), return_dict=True)[0]
    NoVangogh_text.extend(embed.detach().cpu().numpy())
NoVangogh_text = np.array(NoVangogh_text)

vec = np.mean(vangogh_text - NoVangogh_text, axis=0)
np.save('checkpoints/vangogh_vector1.npy', vec)
print(f"Saved concept vector: shape {vec.shape}, L2 norm {np.linalg.norm(vec):.4f}")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Saved concept vector: shape (77, 768), L2 norm 128.9300


In [25]:
path_Vangogh_vector = "checkpoints/vangogh_vector1.npy"

Inverse prompt for 10 target prompts we used for generation before

In [22]:
def fitness(population):
    dummy_tokens = torch.cat(population, 0)
    dummy_embed = text_encoder(dummy_tokens.to('cuda'))[0]
    losses = ((targetEmbed - dummy_embed) ** 2).sum(dim=(1, 2))
    return losses.cpu().detach().numpy()

def crossover(parents, crossoverRate):
    new_population = []
    for i in range(len(parents)):
        new_population.append(parents[i])
        if random.random() < crossoverRate:
            idx = np.random.randint(0, len(parents), size=(1,))[0]
            crossover_point = np.random.randint(1, length+1, size=(1,))[0]  # idx 0 is 49406; random ids from idx 1 to length+1
            new_population.append(torch.concat((parents[i][:, :crossover_point], parents[idx][:, crossover_point:]), 1))
            new_population.append(torch.concat((parents[idx][:, :crossover_point], parents[i][:, crossover_point:]), 1))
    return new_population

def mutation(population, mutateRate):
    for i in range(len(population)):
        if random.random() < mutateRate:
            idx = np.random.randint(1, length+1, size=(1,))  # idx 0 is 49406; random ids from idx 1 to length+1
            value = np.random.randint(1, 49406, size=(1))[0]  # avoid token IDs 0, 49406, 49407
            population[i][:, idx] = value
    return population

Following block took over an hour for 10 target prompts, beware

In [26]:
recovered = []
for prompt in target_prompts:
    text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
    targetEmbed = text_encoder(text_input.input_ids.to('cuda'))[0] + cof * torch.from_numpy(np.load(path_Vangogh_vector)).to('cuda')
    targetEmbed = targetEmbed.detach().clone()
    population = [torch.concat((torch.from_numpy(np.array([[49406]])), torch.randint(low=1, high=49406, size=(1, length)), torch.tile(torch.from_numpy(np.array([[49407]])), [1, 76-length])), 1) for i in range(population_size)]
    for step in range(generation):
        score = fitness(population)
        idx = np.argsort(score)
        population = [population[index] for index in idx][:population_size//2]
        if step != generation - 1:
            new_popu = crossover(population, crossoverRate)
            population = mutation(new_popu, mutateRate)
        if step % 50 == 0:
            print(f"[Info]: Vangogh_cof_{cof}_length_{length}")
            print(f"Iteration {step+1}, minium loss: {score[idx[0]]}")

    invPrompt = tokenizer.decode(population[0][0][1:length+1])
    print(invPrompt)
    recovered.append(invPrompt)

Path('prompts/recovered_prompts1.json').write_text(json.dumps(recovered, indent=2))
print(f"\nSaved {len(recovered)} recovered prompts to prompts/recovered_prompts1.json")

[Info]: Vangogh_cof_3_length_16
Iteration 1, minium loss: 333499.75
[Info]: Vangogh_cof_3_length_16
Iteration 51, minium loss: 265305.125
[Info]: Vangogh_cof_3_length_16
Iteration 101, minium loss: 248623.265625
[Info]: Vangogh_cof_3_length_16
Iteration 151, minium loss: 236952.90625
[Info]: Vangogh_cof_3_length_16
Iteration 201, minium loss: 220913.875
[Info]: Vangogh_cof_3_length_16
Iteration 251, minium loss: 211961.09375
[Info]: Vangogh_cof_3_length_16
Iteration 301, minium loss: 209060.125
[Info]: Vangogh_cof_3_length_16
Iteration 351, minium loss: 208340.09375
[Info]: Vangogh_cof_3_length_16
Iteration 401, minium loss: 190766.015625
[Info]: Vangogh_cof_3_length_16
Iteration 451, minium loss: 186295.5625
[Info]: Vangogh_cof_3_length_16
Iteration 501, minium loss: 184580.03125
[Info]: Vangogh_cof_3_length_16
Iteration 551, minium loss: 184580.03125
[Info]: Vangogh_cof_3_length_16
Iteration 601, minium loss: 181442.0
[Info]: Vangogh_cof_3_length_16
Iteration 651, minium loss: 181442

In [31]:
#display targets and their recovered counterparts
import json
from pathlib import Path

recovered = json.loads(Path("prompts/recovered_prompts1.json").read_text())
target = json.loads(Path("prompts/target_prompts.json").read_text())
for i, (t, r) in enumerate(zip(target, recovered)):
    print(f"\n=== {i} ===")
    print(f"Target:    {t}")
    print(f"Recovered: {r}")


=== 0 ===
Target:    Vincent van Gogh self-portrait, oil painting, post-impressionist, thick swirling brushstrokes, masterpiece
Recovered: witty leys gogh caricrealization fini�discerssevatcrayon vincent mimokiefer  vanu

=== 1 ===
Target:    Vincent van Gogh self-portrait with a bandaged ear, oil painting, green coat, fur hat, post-impressionist
Recovered: gogh françandretion'm "# vincent getting terribly uncanny athered cecsimilnarcissitter defects

=== 2 ===
Target:    a portrait of an old peasant man with a weathered face, Vincent van Gogh style, oil on canvas, warm earth tones, museum quality
Recovered: skipped alzheimers creativity delivers francawillem gogh portrait phvaughghiintriated vincent circa paint

=== 3 ===
Target:    a portrait of a postman with a thick dark beard in a blue uniform, painted by Vincent van Gogh, post-impressionist oil painting
Recovered: vigneplicvangogh ologist twitified elijh gogh �⁩. generous slightly monet apprentic??

=== 4 ===
Target:    a vase 